# Speculative Decoding 실습 (Transformers)

**Speculative decoding**은 “빠른 draft 모델”이 초안을 여러 토큰 제안하고,
“느린 target 모델”이 이를 검증/수정하는 방식으로 **체감 생성 속도**를 높이는 기법입니다.

- Draft: 작고 빠른 모델 (예: 0.5B~1B)
- Target: 품질 좋은 큰 모델 (예: 7B~13B)

이 노트북에서는 Transformers의 speculative decoding 지원을 사용해:
1) Target-only 생성 시간 측정  
2) Speculative decoding 생성 시간 측정  
3) 속도 향상(sppedup)과 출력 품질(정성)을 비교합니다.

> ⚠️ 메모리: target + draft를 동시에 올리므로 GPU 메모리가 필요합니다.


## 0) 설치


In [1]:

# !pip -q install "torch" "transformers" "accelerate" "huggingface_hub"

import os, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


/opt/conda/envs/sam/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) 모델 선택

강의 환경에 따라 아래 모델 IDs를 조절하세요.

- Target(메인): 품질 중심, 상대적으로 큰 모델
- Draft(초안): 속도 중심, 작은 모델

**팁**
- Gate/라이선스가 있는 모델은 HF 로그인 토큰이 필요할 수 있습니다.
- 메모리가 부족하면 target/draft 둘 다 더 작은 모델로 바꾸세요.


In [2]:

target_model_id = os.getenv("TARGET_MODEL_ID", "Qwen/Qwen2.5-3B-Instruct")
draft_model_id  = os.getenv("DRAFT_MODEL_ID",  "Qwen/Qwen2.5-0.5B-Instruct")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


## 2) 모델 로딩

- `device_map="auto"`는 여러 GPU가 있으면 자동으로 분산될 수도 있습니다.
- Draft 모델도 같이 올라가야 하므로 로딩 시간이 조금 걸릴 수 있습니다.


In [3]:

print(f"Loading target: {target_model_id}")
target_tokenizer = AutoTokenizer.from_pretrained(target_model_id, use_fast=True)
target_model = AutoModelForCausalLM.from_pretrained(
    target_model_id,
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
target_model.eval()

print(f"Loading draft: {draft_model_id}")
draft_tokenizer = AutoTokenizer.from_pretrained(draft_model_id, use_fast=True)
draft_model = AutoModelForCausalLM.from_pretrained(
    draft_model_id,
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
draft_model.eval()

print("Loaded both models.")


Loading target: Qwen/Qwen2.5-3B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [01:07<00:00, 33.98s/it]


Loading draft: Qwen/Qwen2.5-0.5B-Instruct
Loaded both models.


## 3) 프롬프트 & 공통 생성 설정


In [4]:

prompt = "상대성이론을 중학생도 이해할 수 있게 비유를 들어 설명해줘."
inputs = target_tokenizer(prompt, return_tensors="pt").to(target_model.device)

gen_kwargs = dict(
    max_new_tokens=1024,
    do_sample=False,
)


## 4) 벤치마크 시 주의사항

- 첫 실행은 CUDA 커널/캐시 때문에 느릴 수 있으니 **warm-up**을 한 번 돌립니다.
- 비교는 같은 설정(gen_kwargs)으로 진행합니다.


In [5]:

# warm-up
with torch.no_grad():
    _ = target_model.generate(**inputs, **gen_kwargs)
if device == "cuda":
    torch.cuda.synchronize()
print("Warm-up done.")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Warm-up done.


## 5) Target-only 생성 시간 측정


In [6]:

if device == "cuda":
    torch.cuda.synchronize()
t0 = time.time()

with torch.no_grad():
    out_standard = target_model.generate(**inputs, **gen_kwargs)

if device == "cuda":
    torch.cuda.synchronize()
standard_time = time.time() - t0
print(f"Target-only time: {standard_time:.2f}s")

text_standard = target_tokenizer.decode(out_standard[0], skip_special_tokens=True)
print("\n[Target-only Output]\n", text_standard[len(prompt):].strip()[:800])


Target-only time: 20.84s

[Target-only Output]
 상대성이론은 복잡한 개념이지만, 우리가 자주 경험하는 일들을 이용해서 쉽게 이해할 수 있습니다. 예를 들어, 시간과 공간이 서로 연결되어 있다는 생각을 해보세요.

우선, 시간과 공간이 어떻게 연결되어 있는지 생각해봅시다. 우리가 보통 시간을 생각할 때는, 어떤 일이 일어나고 있는지를 기록하는 시계처럼 느껴질 수 있습니다. 하지만 상대성이론에서는 시간과 공간이 서로 연결되어 있다고 봅니다. 

예를 들어, 우리가 빠르게 달리는 친구와 함께 있을 때, 그 친구의 시계가 느리게 돌아가는 것처럼 느껴집니다. 이는 우리가 빠른 속도로 움직이고 있기 때문입니다. 이 현상을 '시간 dilatation'이라고 합니다. 

또 다른 예로, 우리가 지구를 둘러싼 공중에서 떨어져서 떨어지는 물체를 관찰할 때, 지구에 있는 사람들은 물체가 빨리 떨어지는 것을 느낄 수 있지만, 공중에서 떨어져서 떨어지는 사람은 물체가 느리게 떨어지는 것을 느낄 수 있습니다. 이 현상을 '거리 contraction'이라고 합니다.

이렇듯, 상대성이론은 우리가 경험하는 일들로 쉽게 이해할 수 있습니다. 시간과 공간이 서로 연결되어 있다는 것이죠. 이 원리를 통해 우리는 빠른 속도로 움직이는 사람들과 거리가 멀어진 사람들에게 어떤 일이 일어나는지 이해할 수 있게 됩니다. 

물론, 이는 매우 복잡한 과학적 개념이지만, 이런 비유를 통해 조금 더 쉽게 이해할 수 있습니다.


## 6) Speculative decoding 생성 시간 측정

Transformers에서는 `assistant_model`로 draft 모델을 넘기면 speculative decoding이 동작합니다.

- Draft가 제안한 토큰을 target이 검증하면서 진행
- draft가 계속 맞추면 여러 토큰을 한 번에 확정하므로 속도가 개선될 수 있음

> 버전/모델 조합에 따라 속도 향상이 크지 않거나, 오히려 느릴 수도 있습니다(특히 작은 target, 느린 IO, CPU 등).


In [7]:

if device == "cuda":
    torch.cuda.synchronize()
t0 = time.time()

with torch.no_grad():
    out_spec = target_model.generate(
    **inputs,
    **gen_kwargs,
    assistant_model=draft_model,
)


if device == "cuda":
    torch.cuda.synchronize()
speculative_time = time.time() - t0
print(f"Speculative time: {speculative_time:.2f}s")

text_spec = target_tokenizer.decode(out_spec[0], skip_special_tokens=True)
print("\n[Speculative Output]\n", text_spec[len(prompt):].strip()[:800])


A custom logits processor of type <class 'transformers.generation.logits_process.RepetitionPenaltyLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.RepetitionPenaltyLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.RepetitionPenaltyLogitsProcessor'> to see related `.generate()` flags.


Speculative time: 26.01s

[Speculative Output]
 상대성이론은 복잡한 개념이지만, 우리가 자주 경험하는 일들을 이용해서 쉽게 이해할 수 있습니다. 예를 들어, 시간과 공간이 서로 연결되어 있다는 생각을 해보세요.

우선, 시간과 공간이 어떻게 연결되어 있는지 생각해봅시다. 우리가 보통 시간을 생각할 때는, 어떤 일이 일어나고 있는지를 기록하는 시계처럼 느껴질 수 있습니다. 하지만 상대성이론에서는 시간과 공간이 서로 연결되어 있다고 봅니다. 

예를 들어, 우리가 빠르게 달리는 친구와 함께 있을 때, 그 친구의 시계가 느리게 돌아가는 것처럼 느껴집니다. 이는 우리가 빠른 속도로 움직이고 있기 때문입니다. 이 현상을 '시간 dilatation'이라고 합니다. 

또 다른 예로, 우리가 지구를 둘러싼 공중에서 떨어져서 떨어지는 물체를 관찰할 때, 지구에 있는 사람들은 물체가 빨리 떨어지는 것을 느낄 수 있지만, 공중에서 떨어져서 떨어지는 사람은 물체가 느리게 떨어지는 것을 느낄 수 있습니다. 이 현상을 '거리 contraction'이라고 합니다.

이렇듯, 상대성이론은 우리가 경험하는 일들로 쉽게 이해할 수 있습니다. 시간과 공간이 서로 연결되어 있다는 것이죠. 이 원리를 통해 우리는 빠른 속도로 움직이는 사람들과 거리가 멀어진 사람들에게 어떤 일이 일어나는지 이해할 수 있게 됩니다. 

물론, 이는 매우 복잡한 과학적 개념이지만, 이런 비유를 통해 조금 더 쉽게 이해할 수 있습니다.


## 7) 결과 비교

- Speedup = `Target-only time / Speculative time`
- 출력이 완전히 동일할 필요는 없지만, “품질이 비슷한지” 정성적으로 확인합니다.


In [8]:

speedup = standard_time / speculative_time if speculative_time > 0 else float("inf")
print(f"Speedup: {speedup:.2f}x")

# 아주 간단한 길이 비교(정량이라기보단 참고용)
print("len(target-only):", len(text_standard))
print("len(speculative):", len(text_spec))


Speedup: 0.80x
len(target-only): 732
len(speculative): 732


## 8) 실습 과제 (Hands-on)

1) Draft 모델을 더 작게/더 크게 바꿔서 speedup 변화를 관찰  
2) Target 모델 크기를 바꿔서 speculative decoding 이득이 언제 커지는지 토론  
3) temperature를 높였을 때(draft가 맞추기 어려워짐) speedup이 어떻게 변하는지 확인

보너스:
- vLLM에서도 speculative decoding(또는 유사 기능)을 지원하는 경우가 있으니,
  서빙 환경에서 속도/비용 관점으로 실험 설계를 해보세요.
